In [5]:
import imaplib
import email
import ssl

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent / "private" / ".env"
load_dotenv(env_path)

imap_server = os.environ["IITB_IMAP_SERVER"]
email_address = os.environ["IITB_EMAIL"]
access_token = os.environ["IITB_IMAP_ACCESS_TOKEN"]


In [6]:
# 1. Create a modern default SSL context
context = ssl.create_default_context()

# 2. Lower the restriction to allow OpenSSL's standard 'DEFAULT' ciphers
# This bridges the gap with servers requiring legacy key exchanges.
# IITB IMAP server need this
context.set_ciphers('DEFAULT')

context

In [7]:
try:
    print(f"Connecting securely to {imap_server}...")
    
    # 3. Pass the custom context into IMAP4_SSL
    mail = imaplib.IMAP4_SSL(imap_server, 993, ssl_context=context)

    mail.login(email_address, access_token)
    print("Login successful!")
    
    mail.logout()

except ssl.SSLError as ssl_err:
    print(f"SSL Specific Error: {ssl_err}")
except Exception as e:
    print(f"Connection failed: {e}")

Connecting securely to imap.iitb.ac.in...
Login successful!


# Now We check getting all the emails from the server

In [52]:
ignore_keywords = ["LDAP login", "access token", "Newsletter", "Survey", "External", "Applications"]


import re


def is_valid_event_body(body_text):
  text_lower = body_text.lower()

  # 1. Check for location markers (venues, rooms, campus buildings)
  location_keywords = [
      "venue",
      "room",
      "auditorium",
      "hall",
      "lc",
      "vmcc",
      "lab",
      "building",
      "ground",
  ]
  has_location = any(loc in text_lower for loc in location_keywords)

  # 2. Check for time or date indicators
  time_date_keywords = ["am", "pm", "ist", "date:", "time:", "scheduled"]
  has_time_date = any(td in text_lower for td in time_date_keywords)

  # 3. Check for event action verbs/phrases
  action_phrases = [
      "is organizing",
      "will be held",
      "cordially invite",
      "scheduled on",
      "to conduct",
      "upcoming",
  ]
  has_action = any(action in text_lower for action in action_phrases)

  # Decision rule: It needs to mention an action and at least one logistical detail (time or location)
  return has_action and (has_location or has_time_date)

In [ ]:
valid_events = []

try:
    print(f"Connecting securely to {imap_server}...")
    
    mail = imaplib.IMAP4_SSL(imap_server, 993, ssl_context=context)

    mail.login(email_address, access_token)
    print("Login successful!")

    mail.select("INBOX", readonly=True)

    status, messages = mail.search(None, "ALL")
    messages_arr = messages[0].split()
    counter = 0
    for num in messages_arr:
        status, data = mail.fetch(num, "(BODY.PEEK[])")
        msg = email.message_from_bytes(data[0][1])
    
        subject = msg["Subject"] or ""
        date = msg["Date"] or ""
        sender = msg["From"] or ""

        if any(keyword in subject for keyword in ignore_keywords):
            continue #skip this message

        body = ""
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    try:
                        body = part.get_payload(decode=True).decode(
                            "utf-8", errors="ignore"
                        )
                        break
                    except Exception:
                        pass
        else:
            try:
                body = msg.get_payload(decode=True).decode("utf-8", errors="ignore")
            except Exception:
                pass

        if not is_valid_event_body(body): 
            continue

        #valid event
        valid_events.append({
            "subject" : subject, 
            "date": date ,
            "sender": sender,
            "body": body
        })
    

except ssl.SSLError as ssl_err:
    print(f"SSL Specific Error: {ssl_err}")
except Exception as e:
    print(f"Connection failed: {e}")

print(f"loaded ${len(valid_events)} events out of ${len(messages_arr)} total messages.")

Connecting securely to imap.iitb.ac.in...
Login successful!
loaded $36 events out of $976 total messages.


In [64]:
print(valid_events[2]["body"])

If you reply to mails received from this list, please take care **NOT**
to send a copy to student-notices.


Dear All,

We are pleased to invite you to the upcoming PRISM Colloquium hosted at 
IIT Bombay:

Title: Combating Antimicrobial Resistance: Synthetic Biology and 
Next-Generation Phage Therapy

Speaker: Prof. Kiran Kondabagil
Department of Biosciences and Bioengineering, IIT Bombay

🗓️ Date: 30 July 2026
🕚 Time: 11:00 AM
📍 Venue: BSBE Seminar Room A
🔗Register now: 
https://docs.google.com/forms/d/e/1FAIpQLSceDpa_eCeNJIntpPpZ8dFE9P3PJ_rGl6_P1wEIVSbORNVWQA/viewform?usp=header

Abstract

Antibiotic resistance has become a global healthcare crisis, with 
multidrug-resistant (MDR) pathogenic bacteria projected to cause 50 
million deaths by 2050. Although bacteriophage therapy is a promising 
alternative, it suffers from major scientific, regulatory, and 
manufacturing challenges. The scientific challenges include a narrow 
host range and the rapid emergence of phage resistance. As t